In [1]:
import pyspark
import os
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('transaction-counts').getOrCreate()


In [2]:
# Read Stream from Kafka

from pyspark.sql.functions import *

df_raw = (
    spark.readStream
         .format("kafka")
         .option("kafka.bootstrap.servers", "localhost:9092")
         .option("subscribe", "transactions-count1")
         .load()
)

# Kafka gives key/value as binary → convert to string
df = df_raw.selectExpr("CAST(value AS STRING)")

In [3]:
from pyspark.sql.types import *
schema = StructType([
    StructField("txn_id", StringType()),
    StructField("txn_type", StringType()),
    StructField("amount", DoubleType()),
    StructField("timestamp", TimestampType())
])

parsed = (df
    .select(from_json(col("value"), schema).alias("data"))
    .select("data.*"))


In [4]:
# Add Watermark + 10-Minute Window Aggregation
agg = (parsed
    .withWatermark("timestamp", "20 minutes")   # allow late data
    .groupBy(
        window(col("timestamp"), "10 minutes"),
        col("txn_type")
    )
    .count()
    .orderBy("window"))

from pyspark.sql.functions import col

flattened = (agg
    .withColumn("window_start", col("window.start"))
    .withColumn("window_end", col("window.end"))
    .drop("window"))



In [ ]:
# Write Each Batch Using foreachBatch
def write_results(batch_df, batch_id):
    (batch_df
     .coalesce(1)            # easy for dashboard ingestion
     .write
     .mode("overwrite")      # overwrite batch folder
     .csv(f"./data/txn_agg2/window_{batch_id}"))
     #.parquet(f"./data/txn_agg/window_{batch_id}"))

query = (flattened.writeStream
         .outputMode("complete")      # allowed with foreachBatch
         .trigger(processingTime="2 minutes")
         .foreachBatch(write_results)
         .option("checkpointLocation", "./chk/txn_agg2")
         .start())

query.awaitTermination()
